#Chuẩn bị dataset

*   Không cần chạy vì có sẵn dataset ở drive rồi



In [ ]:
import io
import os
import sys
import json
import shutil
from pycocotools.coco import COCO
from tqdm import tqdm
from google.colab import drive

TRAIN_LIMIT = None
VAL_LIMIT = None

# Nguong dien tich annotation toi thieu (px), khop voi coco_dataset.py.
MIN_ANN_AREA = 1024

RAW_ZIP_PATH = "/content/gdrive/MyDrive/COCO_Raw_Data"
CLEAN_DATASET = "/content/coco_clean_splits"
TEMP_EXTRACT = "/content/coco_temp"
ANNOTATIONS_DIR = os.path.join(TEMP_EXTRACT, "annotations")

COCO_URLS = {
    "train2017.zip": "http://images.cocodataset.org/zips/train2017.zip",
    "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
    "annotations_trainval2017.zip": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
}

SPLIT_SPECS = [
    ("train", "train2017", f"{TEMP_EXTRACT}/train2017", TRAIN_LIMIT),
    ("val", "val2017", f"{TEMP_EXTRACT}/val2017", VAL_LIMIT),
]

drive.mount('/content/gdrive')
os.makedirs(RAW_ZIP_PATH, exist_ok=True)
os.makedirs(CLEAN_DATASET, exist_ok=True)

for file_name, url in COCO_URLS.items():
    dst = os.path.join(RAW_ZIP_PATH, file_name)
    if not os.path.exists(dst):
        !wget -q -c {url} -O {dst}
    if not os.path.exists(f"/content/{file_name}"):
        !cp {dst} /content/

if not os.path.exists(TEMP_EXTRACT):
    os.makedirs(TEMP_EXTRACT, exist_ok=True)
    !unzip -q /content/annotations_trainval2017.zip -d {TEMP_EXTRACT}
    !unzip -q /content/train2017.zip -d {TEMP_EXTRACT}
    !unzip -q /content/val2017.zip -d {TEMP_EXTRACT}

# Khoi tao annotation handle ngay tu dau (chung cho ca train/val)
inst_handles = {}


def get_inst_handle(annotation_mode):
    """Load instances_{annotation_mode}.json mot lan va cache lai."""
    if annotation_mode not in inst_handles:
        inst_file = os.path.join(ANNOTATIONS_DIR, f"instances_{annotation_mode}.json")
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()
        inst_handles[annotation_mode] = COCO(inst_file)
        sys.stdout = old_stdout
    return inst_handles[annotation_mode]


def process_coco_split(split_name, annotation_mode, img_dir, limit=None):
    print(f"\nProcessing {split_name} (mask-free, annotations only)...")

    capt_file = os.path.join(ANNOTATIONS_DIR, f"captions_{annotation_mode}.json")
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    coco_text = COCO(capt_file)
    sys.stdout = old_stdout
    coco_mask = get_inst_handle(annotation_mode)

    split_root = os.path.join(CLEAN_DATASET, split_name)
    if os.path.exists(split_root):
        shutil.rmtree(split_root)

    img_out = os.path.join(split_root, "images")
    os.makedirs(img_out, exist_ok=True)

    # Copy annotation file vao chinh split de zip tu chua, khong phu thuoc /content/coco_temp
    ann_src = os.path.join(ANNOTATIONS_DIR, f"instances_{annotation_mode}.json")
    ann_dst = os.path.join(split_root, f"instances_{annotation_mode}.json")
    shutil.copy(ann_src, ann_dst)

    all_img_ids = sorted(coco_mask.getImgIds())

    # Loc NGAY TU NGUON: chi giu nhung anh co it nhat 1 annotation hop le
    # (iscrowd=0, area >= MIN_ANN_AREA) de tranh mask bi "muon" tu anh khac
    # khi anh khong co object nao dat yeu cau.
    valid_img_ids = set()
    for img_id in tqdm(all_img_ids, desc=f"Filter anns {split_name}"):
        ann_ids = coco_mask.getAnnIds(imgIds=img_id)
        if not ann_ids:
            continue
        anns = coco_mask.loadAnns(ann_ids)
        if any(a.get("iscrowd", 0) == 0 and a.get("area", 0) >= MIN_ANN_AREA for a in anns):
            valid_img_ids.add(img_id)

    img_ids = sorted(valid_img_ids)
    if limit is not None:
        img_ids = img_ids[:limit]

    metadata = []
    missing_image = 0   # anh khong ton tai tren dia
    missing_caption = 0 # anh hop le nhung khong co caption

    for img_id in tqdm(img_ids, desc=split_name):
        img_info = coco_mask.loadImgs(img_id)[0]
        file_name = img_info["file_name"]
        src_img_path = os.path.join(img_dir, file_name)

        if not os.path.exists(src_img_path):
            missing_image += 1
            continue

        # Khong con tao mask: chi giu cac annotation id de DataLoader sinh mask luc train.
        ann_text_ids = coco_text.getAnnIds(imgIds=img_id)
        text_anns = coco_text.loadAnns(ann_text_ids)
        caption = text_anns[0]["caption"].strip() if len(text_anns) > 0 else ""

        if not caption:
            missing_caption += 1
            continue

        dst_img_path = os.path.join(img_out, file_name)
        shutil.copy(src_img_path, dst_img_path)

        metadata.append({
            "image_id": img_id,
            "annotation_file": f"instances_{annotation_mode}.json",
            "image_path": f"{split_name}/images/{file_name}",
            "caption": caption,
        })

    with open(os.path.join(split_root, "metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    no_valid_ann = len(all_img_ids) - len(valid_img_ids)
    print(f"\n=== {split_name} report ===")
    print(f"Total images in COCO {annotation_mode}: {len(all_img_ids)}")
    print(f"  - Dropped (no valid annotation >= {MIN_ANN_AREA}px): {no_valid_ann}")
    print(f"  - Dropped (image missing on disk)          : {missing_image}")
    print(f"  - Dropped (no caption)                     : {missing_caption}")
    print(f"  - Kept: {len(metadata)} valid samples (masks generated on the fly during training)")


for split_name, annotation_mode, img_dir, limit in SPLIT_SPECS:
    process_coco_split(split_name, annotation_mode, img_dir, limit)

for split_name in ["train", "val"]:
    zip_name = f"coco_clean_{split_name}.zip"
    split_root = os.path.join(CLEAN_DATASET, split_name)

    if os.path.exists(f"/content/{zip_name}"):
        os.remove(f"/content/{zip_name}")

    %cd {CLEAN_DATASET}
    !zip -r -q /content/{zip_name} {split_name}
    %cd /content

    !cp /content/{zip_name} {RAW_ZIP_PATH}/
    print(f"Saved to Drive: {RAW_ZIP_PATH}/{zip_name}")

print("Done")

#Cài đặt thư viện

In [1]:
# KHỞI TẠO MÔI TRƯỜNG SD 1.5 INPAINTING
import os
import sys
from google.colab import drive
# Mount Google Drive
os.chdir("/content")
drive.mount("/content/gdrive")
# Đường dẫn
DRIVE_WORKSPACE = "/content/gdrive/MyDrive/Train_SD15"
# Cài Python 3.10 + pip
!apt-get update -y > /dev/null
!apt-get install python3.10 python3.10-distutils -y > /dev/null
if not os.path.exists("/content/get-pip.py"):
    !wget -q https://bootstrap.pypa.io/get-pip.py
!python3.10 /content/get-pip.py > /dev/null
# Cài requirements
os.chdir(DRIVE_WORKSPACE)
!python3.10 -m pip install -r requirements/requirements.txt \
    --extra-index-url https://download.pytorch.org/whl/cu118
print("Khởi tạo môi trường xong")

Mounted at /content/gdrive
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 36.7 MB/s  0:00:09
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 62.1 MB/s  0:00:02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.3/905.3 MB 13.8 MB/s  0:00:27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 75.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 40.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#Train

In [ ]:
import os
import sys
import glob
import json
import random
import torch
import gc
import shutil
import yaml
from pathlib import Path
from google.colab import drive
#THÔNG SỐ HYPERPARAMETERS
COCO_LIMIT = None               # Số lượng ảnh muốn lấy từ tập TRAIN
VALID_RATIO = 0.05              # Giữ lại 5% train2017 làm validation
SPLIT_SEED = 42                 # Cố định split, giúp các lần train so sánh được
#KHỞI TẠO ĐƯỜNG DẪN & MÔI TRƯỜNG
drive.mount('/content/gdrive')
DRIVE_WORKSPACE = "/content/gdrive/MyDrive/Train_SD15"
OUTPUT_DIR = os.path.join(DRIVE_WORKSPACE, "output_models")
SD15_INPAINT_ROOT = os.path.join(DRIVE_WORKSPACE, "checkpoints", "sd15-inpainting")
SD15_INPAINT_MODEL = os.path.join(SD15_INPAINT_ROOT, "stable-diffusion-inpainting")
SD15_INPAINT_MODEL_ID = "runwayml/stable-diffusion-inpainting"
TRAIN_CONFIG_PATH = os.path.join(DRIVE_WORKSPACE, "configs", "config_sd15_inpaint.yaml")
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
#Khai báo đường dẫn Repo vào hệ thống
sys.path.insert(0, DRIVE_WORKSPACE)
# Lọc log cảnh báo rác
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore"
#CHUẨN BỊ DỮ LIỆU
print("\nPrepare Data Train")
TRAIN_ZIP_PATH = "/content/gdrive/MyDrive/COCO_Raw_Data/coco_clean_train.zip"
BASE_DATA_DIR = "/content/coco_clean_splits"
DATASET_DIR = os.path.join(BASE_DATA_DIR, "train")
os.makedirs(BASE_DATA_DIR, exist_ok=True)
if not os.path.exists(os.path.join(DATASET_DIR, "metadata.json")):
    !cp "{TRAIN_ZIP_PATH}" /content/train_data.zip
    !unzip -q /content/train_data.zip -d {BASE_DATA_DIR}
    !rm /content/train_data.zip
with open(os.path.join(DATASET_DIR, "metadata.json"), "r") as f:
    full_metadata = json.load(f)
actual_metadata = full_metadata if COCO_LIMIT is None else full_metadata[:COCO_LIMIT]
for item in actual_metadata:
    img_filename = os.path.basename(item["image_path"])
    item["image_path"] = os.path.join(DATASET_DIR, "images", img_filename)
    # Khong con lam mask: giu nguyen annotation_file de DataLoader doc annotation truc tiep.
    item.setdefault("annotation_file", "instances_train2017.json")
# Tách 5% validation cố định; không copy ảnh/mask vì metadata đã dùng đường dẫn tuyệt đối.
rng = random.Random(SPLIT_SEED)
rng.shuffle(actual_metadata)
val_count = int(len(actual_metadata) * VALID_RATIO)
val_metadata = actual_metadata[:val_count]
train_metadata = actual_metadata[val_count:]

# COCODataset đọc metadata_run.json, nên file này chỉ chứa 95% train.
with open(os.path.join(DATASET_DIR, "metadata_run.json"), "w") as f:
    json.dump(train_metadata, f, indent=4)

VAL_DIR = os.path.join(BASE_DATA_DIR, "validation_from_train")
os.makedirs(VAL_DIR, exist_ok=True)
# Annotation file can cho validation dataset doc truc tiep.
_ann_for_val = os.path.join(DATASET_DIR, "instances_train2017.json")
if os.path.exists(_ann_for_val):
    shutil.copy(_ann_for_val, os.path.join(VAL_DIR, "instances_train2017.json"))
with open(os.path.join(VAL_DIR, "metadata_run.json"), "w") as f:
    json.dump(val_metadata, f, indent=4)

print(f"Train: {len(train_metadata):,} mẫu | Validation: {len(val_metadata):,} mẫu")
#TẢI MODEL SD 1.5 INPAINTING NẾU CHƯA CÓ
print("\nCheck SD 1.5 Inpainting model...")
if not os.path.isdir(SD15_INPAINT_MODEL):
    from huggingface_hub import snapshot_download
    print(f"Downloading {SD15_INPAINT_MODEL_ID} -> {SD15_INPAINT_MODEL}")
    snapshot_download(repo_id=SD15_INPAINT_MODEL_ID, local_dir=SD15_INPAINT_MODEL)
if not os.path.isdir(SD15_INPAINT_MODEL):
    raise FileNotFoundError(f"Khong tim thay SD 1.5 Inpainting tai: {SD15_INPAINT_MODEL}")
#SYNC CONFIG
with open(TRAIN_CONFIG_PATH, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)
train_cfg["pretrained_model_name_or_path"] = SD15_INPAINT_MODEL
train_cfg["output_dir"] = OUTPUT_DIR
train_cfg["train_data"]["datasets"][0]["data_root"] = DATASET_DIR
# validation_dataset tính validation loss trên 5% dữ liệu đã tách.
train_cfg["validation_dataset"] = {
    "dataset_class": "COCODataset",
    "data_root": VAL_DIR,
    # Validation nay duoc tach TU train split nen annotation file van la cua train.
    "annotation_file": "instances_train2017.json",
}
train_cfg["validation_batch_size"] = train_cfg.get("train_batch_size", 2)
# Check validation loss cùng nhịp với checkpoint để chọn checkpoint tốt nhất.
train_cfg["validation_steps"] = train_cfg.get("checkpointing_steps", 1000)
with open(TRAIN_CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False)
print("Train config synced to local PowerPaint release:")
print({
    "pretrained_model_name_or_path": train_cfg["pretrained_model_name_or_path"],
    "output_dir": train_cfg["output_dir"],
    "train_data_root": train_cfg["train_data"]["datasets"][0]["data_root"],
    "validation_data_root": train_cfg["validation_dataset"]["data_root"],
    "validation_steps": train_cfg["validation_steps"],
})
#KHỞI CHẠY TRAINING
#Dọn dẹp VRAM trước khi chạy
torch.cuda.empty_cache(); gc.collect()
#Logic tìm và chạy tiếp từ file lưu (Resume)
ckpt_list = glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*"))
resume_arg = ""

print("\nKiểm tra Checkpoint...")
if ckpt_list:
    # Parse checkpoint-<step> và chọn step lớn nhất (không dựa vào thời gian filesystem).
    steps = []
    for p in ckpt_list:
        name = os.path.basename(p)
        try:
            steps.append((int(name.split("-")[1]), p))
        except (IndexError, ValueError):
            continue
    if steps:
        latest_step, latest_ckpt = max(steps, key=lambda x: x[0])
        resume_arg = f'--resume_from_checkpoint="{os.path.basename(latest_ckpt)}"'
        print(f"tìm thấy bản lưu step {latest_step}: {latest_ckpt}")
    else:
        print("Không tìm thấy bản lưu hợp lệ")
else:
    print("Không tìm thấy bản lưu")
os.chdir(DRIVE_WORKSPACE)
!python3.10 train_sd15_inpaint.py --config "configs/config_sd15_inpaint.yaml" {resume_arg}
    # --max_train_steps=500
    # --checkpointing_steps=250

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

Prepare Data Train
